# 01 — Nguồn OHLCV và kiểm định

Dự báo log return phiên kế tiếp, sau đó suy ra giá và hướng. Chạy các ô từ trên xuống. Dữ liệu dừng tại 30/09/2025.

## Chuẩn bị môi trường

Trên máy cá nhân, cài `pip install -r requirements.txt` trước. Các lệnh mặc định dùng snapshot được lưu trong dự án; không tự thay dữ liệu bằng lần tải mạng mới.

In [ ]:
# Tìm thư mục gốc khi chạy từ repo, từ notebooks/ hoặc trên Google Colab.
from pathlib import Path
import sys, subprocess, json, os
candidates = [Path.cwd(), Path.cwd().parent, Path('/content/fpt-stock-trends-prediction')]
ROOT = next((p for p in candidates if (p/'src/train.py').exists()), None)
if ROOT is None and 'google.colab' in sys.modules:
    subprocess.run(['git','clone','https://github.com/thiemhoang4807/fpt-stock-trends-prediction.git','/content/fpt-stock-trends-prediction'], check=True)
    ROOT = Path('/content/fpt-stock-trends-prediction')
if ROOT is None:
    raise FileNotFoundError('Hãy mở notebook trong repository đã tải về.')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT/'src'))
if 'google.colab' in sys.modules:
    subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
import pandas as pd
from IPython.display import display, Image
print('Thư mục làm việc:', ROOT)


## Sáu cột OHLCV

`time`: phiên; `open/high/low/close`: giá mở/cao/thấp/đóng, nghìn VND/cổ phiếu; `volume`: số cổ phiếu. Nguồn giá là VCI qua vnstock. Snapshot giúp giữ cùng phiên bản dữ liệu giữa các lần chạy.

In [ ]:
from fetch_data import main as fetch
assert fetch(['--verify']) == 0
prices=pd.read_csv(ROOT/'data/raw/fpt/fpt_historical_price.csv')
display(prices.head())
display(prices.tail())

## Khi cần tải trực tuyến

Chủ động chạy `python src/fetch_data.py --download --start 2007-01-01 --end 2025-09-30`. Tải lại có thể thay phiên bản giá điều chỉnh; phải chạy lại toàn pipeline. Notebook này chỉ xác minh snapshot hiện có.

In [ ]:
display(pd.DataFrame({'dtype':prices.dtypes.astype(str),'missing':prices.isna().sum()}))